### I am trying to use chromadb on a large IMDB Dataset


### 1. Load the dataset
### Download the CSV from Kaggle (or use kagglehub/opendatasets if you want it scripted), then load it.

In [1]:
import pandas as pd

df = pd.read_csv("imdb_movies.csv")

print(df.columns)
print(df.shape)
df.head()

Index(['names', 'date_x', 'score', 'genre', 'overview', 'crew', 'orig_title',
       'status', 'orig_lang', 'budget_x', 'revenue', 'country'],
      dtype='object')
(10178, 12)


,names,date_x,score,genre,overview,crew,orig_title,status,orig_lang,budget_x,revenue,country
0,Creed III,03/02/2023,73.0,"Drama, Action","After dominating the boxing world, Adonis Cree...","Michael B. Jordan, Adonis Creed, Tessa Thompso...",Creed III,Released,English,75000000.0,2.716167e+08,AU
1,Avatar: The Way of Water,12/15/2022,78.0,"Science Fiction, Adventure, Action",Set more than a decade after the events of the...,"Sam Worthington, Jake Sully, Zoe Saldaña, Neyt...",Avatar: The Way of Water,Released,English,460000000.0,2.316795e+09,AU
2,The Super Mario Bros. Movie,04/05/2023,76.0,"Animation, Adventure, Family, Fantasy, Comedy","While working underground to fix a water main,...","Chris Pratt, Mario (voice), Anya Taylor-Joy, P...",The Super Mario Bros. Movie,Released,English,100000000.0,7.244590e+08,AU
3,Mummies,01/05/2023,70.0,"Animation, Comedy, Family, Adventure, Fantasy","Through a series of unfortunate events, three ...","Óscar Barberán, Thut (voice), Ana Esther Albor...",Momias,Released,"Spanish, Castilian",12300000.0,3.420000e+07,AU
4,Supercell,03/17/2023,61.0,Action,Good-hearted teenager William always lived in ...,"Skeet Ulrich, Roy Cameron, Anne Heche, Dr Quin...",Supercell,Released,English,77000000.0,3.409420e+08,US


### Typical columns in this dataset: names, date_x, score, genre, overview, crew, orig_title, status, orig_lang, budget_x, revenue, country.

### The overview column (plot summary) is what you'll embed — it's the semantically rich text, similar to how you used the review column for Amazon.

### Step:2 Clean it up

In [2]:
df = df.dropna(subset=['overview'])
df = df.drop_duplicates(subset=['names']) 
df = df.reset_index(drop=True)

# Optional: cap dataset size while testing
df = df.head(2000)  # remove/increase once it works end-to-end

### Step:3. Build the ChromaDB collection

In [5]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="./chroma_imdb_db")

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="imdb_movies_collection",
    embedding_function=embed_fn
)

### Step:4. Insert in batches (important for larger datasets)

#### Chroma can choke if you try to add tens of thousands of rows in one call, so batch it:

In [6]:
batch_size = 200

for start in range(0, len(df), batch_size):
    end = start + batch_size
    batch = df.iloc[start:end]

    documents = batch["overview"].astype(str).tolist()
    ids = [f"movie_{i}" for i in batch.index]
    metadatas = [
        {
            "title": row["names"],
            "genre": str(row.get("genre", "")),
            "score": float(row["score"]) if pd.notna(row.get("score")) else 0.0,
            "release_date": str(row.get("date_x", "")),
        }
        for _, row in batch.iterrows()
    ]

    collection.add(documents=documents, ids=ids, metadatas=metadatas)
    print(f"Inserted rows {start}–{end}")

print("Total in collection:", collection.count())

Inserted rows 0–200
Inserted rows 200–400
Inserted rows 400–600
Inserted rows 600–800
Inserted rows 800–1000
Inserted rows 1000–1200
Inserted rows 1200–1400
Inserted rows 1400–1600
Inserted rows 1600–1800
Inserted rows 1800–2000
Total in collection: 2000


### Step:5. Simple similarity search function

In [14]:
def search_movies(query, n_results=5):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        print(f"🎬 {meta['title']} (score: {meta['score']}) | dist: {dist:.4f}")
        print(f"   {doc[:150]}...")
        print()

# Try it
search_movies("Best action movies with high ratings")

🎬 X-Rated: The Greatest Adult Movies of All Time (score: 63.0) | dist: 0.5816
   The evolution of adult cinema through the most influential films in history, a journey that begins in the 1970s and ends nowadays. An in-depth analysi...

🎬 Avatar: The Deep Dive - A Special Edition of 20/20 (score: 75.0) | dist: 0.5902
   An inside look at one of the most anticipated movie sequels ever with James Cameron and cast....

🎬 The Adventures of Sharkboy and Lavagirl (score: 51.0) | dist: 0.6142
   Everyone always knew that Max had a wild imagination, but no one believed that his wildest creations -- a boy raised by watchful great white sharks an...

🎬 X (score: 68.0) | dist: 0.6277
   In 1979, a group of young filmmakers set out to make an adult film in rural Texas, but when their reclusive, elderly hosts catch them in the act, the ...

🎬 Princess Principal Crown Handler: Chapter 2 (score: 73.0) | dist: 0.6300
   The second out of six movie sequels to the TV series....

